## NLP Final
### Part 2: Topic Detection - BERTopic (Title)
### Aren Mizuno
### March 12, 2026

In [1]:
# Imports + install
!pip -q install bertopic sentence-transformers umap-learn hdbscan

import numpy as np
import pandas as pd
import torch
import re

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from google.colab import drive, files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 14.7 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [2]:
# Cuda
torch.cuda.is_available()

True

In [3]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# Mount drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# Load the cleaned dataset
data_path = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/news_final_project_cleaned.parquet"
df = pd.read_parquet(data_path, engine="pyarrow")
df.head(2)

,url,date,title,text
0,https://blockworks.co/price/bad,2025-06-23,"Bad Idea AI Price (BAD), Market Cap, Price Tod...","Bad Idea AI Price (BAD), Market Cap, Price Tod..."
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...


In [6]:
# Check shape
df.shape

(136233, 4)

In [7]:
# Text cleaning helpers
TOPIC_JUNK_PHRASES = [
    "open menu", "menu", "search", "login", "log in", "sign up", "sign in",
    "subscribe", "privacy policy", "terms of service", "terms and conditions",
    "contact us", "contact support", "newsletter", "skip to content",
    "follow us", "read the rules", "mobile app", "read more", "click here",
    "all rights reserved", "advertisement", "sponsored content",
    "yahoo finance", "reuters", "associated press", "ap news",
]

TOPIC_JUNK_WORDS = {
    "home", "world", "business", "finance", "sports", "weather",
    "entertainment", "store", "blog", "forums", "shop", "mail", "more"
}


def clean_text(s):
    s = str(s).lower()

    # remove phrase-level boilerplate first
    for phrase in TOPIC_JUNK_PHRASES:
        s = re.sub(rf"\b{re.escape(phrase)}\b", " ", s)

    # remove urls
    s = re.sub(r"http\S+|www\.\S+", " ", s)

    # remove timestamps like 23:08:29 PKT or 12:45 PM
    s = re.sub(r"\b\d{1,2}:\d{2}(?::\d{2})?\s*(?:am|pm|[A-Z]{2,5})?\b", " ", s)

    # remove dates like 2024-07-01 or 07/01/2024
    s = re.sub(r"\b\d{4}-\d{2}-\d{2}\b", " ", s)
    s = re.sub(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", " ", s)

    # remove ticker fragments like [BFRG] or $NVDA
    s = re.sub(r"\[[A-Z0-9.\-]{1,10}\]", " ", s)
    s = re.sub(r"\$[A-Za-z]{1,6}\b", " ", s)

    # remove weird concatenated site-chrome tokens
    s = re.sub(r"\b[A-Za-z]+(?:[A-Z][a-z]+){2,}\b", " ", s)

    # remove separator junk
    s = re.sub(r"[_|•\-–—/]+", " ", s)

    # keep only letters/spaces
    s = re.sub(r"[^a-z\s]", " ", s)

    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()

    # remove single-word junk after normalization
    words = [w for w in s.split() if w not in TOPIC_JUNK_WORDS]

    # optional: drop 1-character tokens
    words = [w for w in words if len(w) > 1]

    return " ".join(words)

df["title"] = df["title"].map(clean_text)

In [8]:
# Set up embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
# UMAP reduction
umap_model = UMAP(
    n_components=5,
    n_neighbors=100,
    min_dist=0.1,
    metric="cosine",
    random_state=0
)


In [10]:
# HDBSCAN clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=500,
    min_samples=50,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [11]:
# c-TF-IDF vectorizer
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=30_000    # prevents vocab explosion
)

In [12]:
# Topic representation
ctfidf_model = ClassTfidfTransformer()

In [13]:
# BERTopic model
bert_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    verbose=True
)

# Fit + transform
docs = df["title"].astype(str).tolist()
topics, probs = bert_model.fit_transform(docs)
df["bertopic_topic_title"] = topics

2026-03-10 23:54:29,643 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/4258 [00:00<?, ?it/s]

2026-03-10 23:55:06,077 - BERTopic - Embedding - Completed ✓
2026-03-10 23:55:06,079 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 00:04:25,198 - BERTopic - Dimensionality - Completed ✓
2026-03-11 00:04:25,202 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 00:04:38,980 - BERTopic - Cluster - Completed ✓
2026-03-11 00:04:39,014 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 00:04:43,082 - BERTopic - Representation - Completed ✓


In [14]:
# Number of topics
topic_info = bert_model.get_topic_info()
len(topic_info)

5

In [15]:
# Topic summary
topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1251,-1_bahamas_stew_native stew_bahamas ai,"[bahamas, stew, native stew, bahamas ai, stew ...",[native stew bahamas ai art photos videos baha...
1,0,131886,0_ai_news_chatgpt_new,"[ai, news, chatgpt, new, intelligence, openai,...",[from ai to ai why the magic box of artificial...
2,1,1502,1_salary_levels fyi_fyi_levels,"[salary, levels fyi, fyi, levels, scientist sa...","[mantech data scientist salary levels fyi, sag..."
3,2,1062,2_le_ai_fa_ka,"[le, ai, fa, ka, sa, na, ai le, metatrader, te...",[fa asala le ave ta avale na maliliu ai tagata...
4,3,532,3_philanthropic_new philanthropic_foundation_com,"[philanthropic, new philanthropic, foundation,...",[nemacolin owner maggie hardy philanthropic he...


In [16]:
# Save as parquet
save_path = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/bertopic_info_title.parquet"
topic_info.to_parquet(save_path, engine="pyarrow", compression="snappy")
print("Saved to:", save_path)

Saved to: /content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/bertopic_info_title.parquet


In [17]:
df.head(5)

,url,date,title,text,bertopic_topic_title
0,https://blockworks.co/price/bad,2025-06-23,bad idea ai price bad market cap price today c...,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",0
1,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,this ai video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,0
2,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,if using ai feels like chore try this boing boing,"If using AI feels like a chore, try this - Boi...",0
3,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,the road ahead how china ai foundation model i...,The Road Ahead: How China's AI Foundation Mode...,0
4,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,microsoft and nvidia to empower developers wit...,Microsoft and Nvidia to Empower Developers wit...,0


In [18]:
# Save as parquet
save_path = "/content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/bertopic_title.parquet"
df.to_parquet(save_path, engine="pyarrow", index=False)
print("Saved to:", save_path)

Saved to: /content/drive/MyDrive/UChicago/Masters/Winter/NLP/Final Project/parquets/bertopic_title.parquet
